# 01 — Build Arcadia v2 Dataset

Phase A (VCF -> SNP matrix) + Phase B (labels, phylogroups, splits, PCA baseline). Thin by design: every cell below calls into `src/amr_data`; no business logic lives in this notebook. See `docs/PIPELINE.md` for the full stage diagram.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.amr_data.config import load_paths, load_params
from src.amr_data.io import load_pickle, save_pickle

paths = load_paths()
params = load_params()
print(f'Environment: {paths.env}')

## Phase A — VCF to SNP matrix

Requires the raw VCF (Zenodo 12692732, see `docs/DATA_PROVENANCE.md`). Skip this cell if `arcadia_v2_snp_matrix.pkl` already exists at `paths.processed('snp_matrix_pkl')`.

In [ ]:
from src.amr_data.vcf_to_snp import stream_vcf_to_snp_matrix, VcfToSnpConfig

snp_pkl_path = paths.processed('snp_matrix_pkl')
if snp_pkl_path.exists():
    print(f'Already built: {snp_pkl_path}')
else:
    result = stream_vcf_to_snp_matrix(
        vcf_path=paths.raw('vcf_filtered'),
        sra_map_path=paths.raw('sra_to_genome_map'),
        pheno_path=paths.raw('phenotype_matrix'),
        cfg=VcfToSnpConfig(**params['vcf_to_snp']),
    )
    save_pickle(vars(result), snp_pkl_path)
    print(f'Saved: {snp_pkl_path}')

## Phase B — labels, phylogroups, splits

Requires `strains_metadata.csv` and the phenotype matrix (same Zenodo record as the VCF).

In [ ]:
import pandas as pd
from src.amr_data.labels import build_label_matrix
from src.amr_data.phylogroup import derive_phylogroup, build_cluster_labels
from src.amr_data.splits import (
    random_split, phylogroup_stratified_split, lineage_holdout_split,
    groups_from_phylogroup, save_split_diagnostics,
)

snp_data = load_pickle(snp_pkl_path)
pheno = pd.read_csv(paths.raw('phenotype_matrix'))
metadata = pd.read_csv(paths.raw('strains_metadata'))

samples_df = pd.DataFrame({'Genome.Name': snp_data['selected_genome_names']})
samples_df = samples_df.merge(pheno, on='Genome.Name', how='left')
samples_df = samples_df.merge(metadata[['Genome.Name', 'MLST', 'Other.Typing']],
                               on='Genome.Name', how='left')

Y, label_mask = build_label_matrix(samples_df)
phylogroup_final = derive_phylogroup(samples_df['Other.Typing'], samples_df['MLST'])
cluster_labels, phylogroup_names, pg_to_int = build_cluster_labels(phylogroup_final)

amp_idx = params['labels']['antibiotics'].index('AMP')
splits = {
    'random_split': random_split(Y, amp_idx, seed=params['seed']),
    'phylogroup_stratified_split': phylogroup_stratified_split(cluster_labels, seed=params['seed']),
    'lineage_holdout_split': lineage_holdout_split(groups_from_phylogroup(cluster_labels), seed=params['seed']),
}
diag = save_split_diagnostics(
    splits, Y, label_mask, params['labels']['antibiotics'],
    paths.results_dir / 'splits' / 'split_diagnostics.csv',
)
diag

## PCA-1024 baseline (B8: sparse, memory-safe)

In [ ]:
from src.amr_data.baselines import build_pca_features

rs = splits['random_split']
Xtr, Xva, Xte, pca, scaler = build_pca_features(
    snp_data['X_snp'], rs['train_idx'], rs['val_idx'], rs['test_idx'],
)
print(f'Variance retained: {pca.explained_variance_ratio_.sum():.4f}')